# 쿠션톡 AI 판단기 학습 (Colab 무료 T4)

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르고 저장
2. 위에서부터 칸을 하나씩 실행 (▶ 또는 Shift+Enter)
3. 끝나면 구글 드라이브 `MyDrive/cushion_judge` 에 모델 · 실험 결과가 생긴다 → 받아서 `ml/models/judge` 에 둔다

API 키(.env)는 필요 없다. 이 노트북에 올리지 말 것.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv  # Tesla T4 가 보여야 함

## 1. 코드 · 데이터 받기

In [ ]:
BRANCH = 'feat/etc-ai-local-judge'  # main에 합쳐졌으면 'main'
!rm -rf Adventure-Design && git clone -q -b $BRANCH https://github.com/tacocat404/Adventure-Design.git
%cd Adventure-Design
!pip install -q transformers==5.22.1
!ls ml/data

## 2. 학습 (T4 약 5분)
파인튜닝 4에폭 → kNN 뱅크 → 스태킹. 결과: `ml/models/judge`

In [ ]:
!python ml/train.py

## 3. 발표용 실험 (T4 약 10분)
구성 요소별 기여 + 학습 곡선(500 · 1000 · 2000 · 전체). 결과: `ml/experiments.md`

In [ ]:
!python ml/experiments.py

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('ml/experiments.md', encoding='utf-8').read()))

## 4. 실패 상황 시뮬레이션 · 평가 (선택)
평가의 API 비교 줄은 키가 없어서 빠진다 (로컬 판단만).

In [ ]:
!python ml/stress_test.py > /dev/null && head -40 ml/stress_results.md
!python ml/evaluate.py > /dev/null && head -30 ml/results.md

## 5. 구글 드라이브에 저장
세션이 끊기면 파일이 지워지므로 꼭 저장한다. 모델 약 440MB.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cushion_judge
!cp -r ml/models/judge /content/drive/MyDrive/cushion_judge/
!cp ml/experiments.md ml/experiments.json ml/stress_results.md ml/results.md /content/drive/MyDrive/cushion_judge/ 2>/dev/null
!ls -la /content/drive/MyDrive/cushion_judge /content/drive/MyDrive/cushion_judge/judge